# 第一章：serdes历史

并行传输与串行传输

SerDes是Serializer/Deserializer的缩写，即串行器和解串器，顾名思义是一种将并行数据转换成串行数据发送，将接收的串行数据转换成并行数据的”器件“。对于FPGA工程师来说“串并转换”再熟悉过不过了，只不过SerDes是一种需要数模硬件实现的，用于高速传输的“高级”串并转换器件。至于接口从最初从串口到并口，再回归到串口的历史发展，可以阅读相关的文献，借此可以了解一下系统同步、源同步的技术瓶颈。目前，商用基于SerDes架构的通信协议最高可实现单通道56Gbps（好像已经可达112Gbps）的速率，在未来高带宽、低成本的应用领域会越来越广泛。

SerDes主要具有以下优点：

-   减少布线冲突（串行，并且无单独的时钟线，时钟嵌入在数据流中，从而也解决了限制数据传输速率的信号时钟偏移问题）；
-   抗噪声、抗干扰能力强（差分传输）；
-   降低开关噪声；
-   扩展能力强；
-   更低的功耗和封装成本；


# 接口技术大比拼：并行 (Parallel) vs. 串行 (Serial)

在数据传输领域，接口的设计决定了系统能跑多快。随着传输速率进入 Gbps 时代，接口技术经历了从“多车道低速”到“单车道超高速”的范式转移。

---

## 1. 并行接口 (Parallel Interface)

并行接口像是一条**多车道高速公路**，数据的每一个比特（Bit）都有自己独立的传输通道，并在同一个时钟脉冲下同时发送。

* **工作原理：** $n$ 个比特在 $n$ 根数据线上同步传输。
* **早期应用：** IDE (PATA) 硬盘接口、打印机并口、早期主板总线。
* **优点：** 逻辑设计简单，不需要复杂的串行/解串逻辑。

### ⚠️ 并行接口的“天花板”
当频率提升到 MHz 甚至 GHz 级别时，并行接口遇到了三大物理限制：
1.  **时钟偏移 (Clock Skew)：** 不同导线长度的微小差异（哪怕是 1mm）会导致数据到达接收端的时间不一致，无法在同一个时钟边沿被正确采样。
2.  **串扰 (Crosstalk)：** 密集的并行线在高频翻转时，强烈的电磁耦合会互相干扰信号。
3.  **引脚压力：** 64 位并行总线需要庞大的引脚数量，极大增加了芯片和 PCB 的成本。



---

## 2. 串行接口 (Serial Interface)

串行接口像是一条**单车道超高速铁路**，数据位按顺序一个接一个地在同一对差分线上高速通过。

* **工作原理：** 数据被“打包”成比特流，按时间顺序排列发送。
* **现代应用：** USB 3.0/4, PCIe, SATA, HDMI, Ethernet。
* **核心技术：** 依赖 **SerDes**（串行器/解串器）进行数据转换。

### 🚀 为什么串行反而更快？
1.  **消除偏移：** 只有一对线传输，不存在多线之间的同步延迟问题。
2.  **嵌入式时钟 (CDR)：** 通过编码（如 8b/10b），时钟信息被隐藏在数据流中。接收端利用 **时钟数据恢复 (CDR)** 自动对齐，彻底解决了相位抖动问题。
3.  **差分技术加持：** 串行通常配合差分信号使用，具有极强的抗噪和低辐射特性。



---

## 3. 核心对比表

| 特性 | 并行接口 (Parallel) | 串行接口 (Serial) |
| :--- | :--- | :--- |
| **传输通道** | 8/16/32/64 根信号线 | 1 对或几对差分线 |
| **同步方式** | 依赖外部公共时钟线 | 嵌入式时钟 (Self-clocking) |
| **布线复杂度** | 极高（需保证几十根线严格等长） | 相对简单（仅需保证差分对内等长） |
| **抗干扰能力** | 弱（线间串扰严重） | 极强（利用共模抑制） |
| **物理尺寸** | 占用大量引脚和空间 | 节省引脚，适合小型化设计 |
| **最高速率** | 受限（通常 < 1 Gbps） | 极高（单通道 112 Gbps+） |

---

## 4. SerDes：连接两个世界的桥梁

实际上，现代芯片**内部是并行的，外部是串行的**。

* **芯片内部：** 为了处理方便，CPU 或 FPGA 内部依然采用宽位宽的并行总线。
* **芯片边缘：** **SerDes** 充当翻译官。它在发送端把 32/64 位的并行数据“压缩”成一路超高速串行流；在接收端再“解压”还原回并行数据。

> **总结：** 串行接口通过牺牲逻辑复杂度（增加 SerDes 电路），换取了物理层传输速率的质跃。


为什么非要用 SerDes？（解决三大难题）   
在学习了基础码制和逻辑电路后，你会发现当频率提升到 GHz 级别时，基础电路会遇到物理极限：

-   解决“时钟偏移” (Clock Skew)：
并行总线如果跑 32 位，只要有一根导线长了一点点，数据到达的时间就不一致。串行传输直接把时钟“藏”在数据里，不存在导线长度差的问题。

-   减少引脚数量：
传输同样带宽的数据，并行可能需要 64 个引脚，而 SerDes 可能只需要 2 对（4 个）差分引脚。这极大地减小了芯片封装的尺寸。

-   克服电磁干扰 (EMI)：
SerDes 采用差分对 (Differential Pair) 传输。两根线信号相反，外界干扰对两根线的影响相同，抵消后信号依然清晰。



# SerDes 信号传输详解：单端 (Single-Ended) vs. 差分 (Differential)

在高速串行器/解串器（SerDes）设计中，信号完整性是核心。理解单端与差分信号的区别，是掌握现代高速接口（如 PCIe, USB 3.0, Ethernet）的基础。

---

## 1. 单端信号 (Single-Ended Signaling)

单端信号是最传统的传输方式，通过一根导线传输信号，并以公共地（GND）作为参考点。

* **硬件结构：** 1 根信号线 + 1 根地线。
* **判断逻辑：** 接收端测量 $V_{signal}$ 与 $V_{gnd}$ 之间的电位差。
* **局限性：**
    * **抗干扰能力弱：** 任何地电平的波动（地弹）或外部电磁干扰（EMI）都会直接叠加在信号上，导致逻辑误判。
    * **回流路径受限：** 高频下，电流寻找回流路径产生的环路面积较大，容易产生辐射干扰。



---

## 2. 差分信号 (Differential Signaling)

差分信号使用一对相位相反、振幅相同的信号线（$P$ 和 $N$）来传输一个逻辑位。

* **硬件结构：** 2 根互补的信号线。
* **判断逻辑：** 接收端只关注两线间的压差 $V_{diff} = V_{pos} - V_{neg}$。
* **核心优势：**
    * **共模抑制（Common-Mode Rejection）：** 环境噪声通常会同时耦合到两根线上，但在接收端求差值时，噪声会被抵消。
    * **低摆幅、高速度：** 由于抗噪性强，逻辑翻转不再需要高电压，可以使用更小的电压摆幅（如 400mV），从而显著提升翻转速度。
    * **低 EMI：** 两线产生的电磁场方向相反，相互抵消，减少了对外界的干扰。



---

## 3. 关键特性对比表

| 特性 | 单端信号 (Single-Ended) | 差分信号 (Differential) |
| :--- | :--- | :--- |
| **信号线数量** | 1 (低密度布线) | 2 (占用空间较大) |
| **抗干扰性能** | 差 | 极佳 |
| **典型电压摆幅** | 高 (1.8V, 3.3V) | 低 (200mV ~ 800mV) |
| **最大速率** | 通常 < 1Gbps | 可达 112Gbps 以上 |
| **布线复杂度** | 低 | 高（需严格匹配阻抗与等长） |
| **应用场景** | I2C, UART, GPIO, DDR 地址线 | **SerDes**, PCIe, SATA, MIPI |

---

## 4. 为什么 SerDes 必须选择差分？

SerDes 技术的目标是在极高频率下传输海量数据，单端信号在此时会面临致命问题：

1.  **地弹 (Ground Bounce)：** 高速翻转产生的瞬态电流会使地电位不稳定，单端系统几乎无法在此环境下生存。
2.  **损耗补偿：** 差分结构配合**自适应均衡 (EQ)** 技术，能更精确地从已经严重畸变的波形中提取逻辑 0 和 1。
3.  **功耗效率：** 差分信号允许更低的驱动电压，在大规模集成电路中能有效降低动态功耗。

---



## serdes 系统的问题：
1.  系统中链路非理想信道。其传输带宽有限（可理解为低通滤波器）， 导致码元时域展宽，引入码间干扰（ISI）
2.  当信道经过链路是，由于信道损耗（插损）、阻抗不连续（反射、回损）、其他信道的干扰（串扰）等，信号完整性受到破坏、信噪比（SNR）减低，导致误码

# 第二章：serdes相关的IP
显示相关： HDMI DP    

以太网相关：华为 HiLink


PCIE、JESD204B都是基于SerDes的协议，用OSI网络分层模型来类比的话，SerDes更接近于物理层，并强调了电气属性，而PCIE和JESD204B相当于涵盖数据链路层、网络层和传输层，所以SerDes通常又被称之为物理层（PHY）器件。实际上和很多人没分清RS232和URAT之间的关系一样。

# 第三章：整体架构

关键字： 锁相环，编码器，串行器，发送器

SerDes有四种架构：1）并行时钟SerDes；2）嵌入式时钟SerDes；3）8b/10b编码SerDes；4）位交错SerDes。（注：至于这4种所谓的架构有什么区别，实现有什么特点，我还没找到专门的文献来说明）。

常见的SerDes架构是一种基于8b/10b编码并且时钟嵌入进数据流中的架构（是不是就是上面2、3的组合？），主要由物理介质相关子层（ PMD）、物理媒介附加子层（PMA）和物理编码子层（ PCS ）所组成。如下图所示：

底色为浅绿色的子模块为PCS层，负责数据流的编码/解码，是标准的可综合CMOS数字逻辑，可以硬逻辑实现，也可以使用FPGA软逻辑实现。

底色为浅褐色子模块是PMA层，是数模混合CML/CMOS电路，负责负责串化/解串化，是理解SerDes区别于并行接口的关键。

PMD是负责串行信号传输的电气块（未画出）。

PLL模块（PLL Block）负责产生SerDes各个模块所需要的时钟信号，并管理这些时钟之间的相位关系。

一个SerDes通常还要具调试能力，例如伪随机码流产生和比对，各种环回测试，控制状态寄存器以及访问接口，LOS检测，眼图测试等，所以还需要相应的功能测试模块。

整个流程可以简述为：

-   **发送（TX）**

    FPGA软逻辑（fabric）送过来的并行信号，通过接口FIFO（Interface FIFO），送给8b/10b编码器（8b/10b encoder）或扰码器（scambler），以避免数据含有过长连“0”或者连“1”，之后送给串行器（Serializer）进行并->串转换。串行数据经过均衡器（equalizer）调理，由驱动器（driver）发送出去。

-   **接收（RX）**

    外部串行信号由线性均衡器（Linear Equalizer）或DFE （Decision Feedback Equalizer，判决反馈均衡）结构均衡器调理，去除一部分确定性抖动（Deterministic jitter）。CDR从数据中恢复出采样时钟，经解串器变为对齐的并行信号。8b/10b解码器（8b/10b decoder）或解扰器（de-scambler）完成解码或者解扰。如果是异步时钟系统(plesio-synchronous system)，在用户FIFO之前还应该有弹性FIFO来补偿频差。

注：我发现系统性的把SerDes细节讲清楚的文献并不多，并且有些SerDes接口芯片并不完全是上面这种架构，在实际过程中，各个厂家可能会根据性能做不同的算法优化，具体的模块可能用到不同的技术，所以不要片面的理解上述架构是Serdes的唯一架构。


## 1. 发送端模块说明

### 1.1 8b/10b编码器

8b/10b编码是由IBM于1983年发明的，旨在解决系统互联以及GB以太网传输的问题。 是指将8b的数据通过某种编码规则扩展成10b，保证数据流中的“0”和“1”数量基本一致，通过降低效率来增加传输的数据恢复的可靠性。8b/10b 主要具有以下优点：

-   **保证直流（DC）平衡（重点）**

    直流平衡是什么意思呢？如上图所示，由于我们的串行链路中会有交流耦合电容，信号频率越高，阻抗越低，反之频率越低，阻抗越高。因此上面的情况，当码型是高频的时候，基本上可以不损耗的传输过去，但是当码型为连续“0”或者“1”的情况时，电容的损耗就很大，导致幅度不断降低，带来的严重后果是无法识别到底是“1”还是“0”。因此编码就是为了尽量把低频的码型优化成较高频的码型，从而保证低损耗的传输过去。

    8b/10b编码方式总输出位数是10个位，其中“0”与“1”出现的次数总共也仅在三种场合存在，分别为“5个位0与5个位1”、“4个位0与6个位1”、“6个位0与4个位1”。有一个8b/10b编码的技术专用语“不均等性（Disparity）”，其涵义就是指10个位中位0与位1出现次数的差。换句话说，8b/10b编码的“Disparity”就仅有“+2”（4个位0与6个位1）、“+0”（5个位0与5个位1）以及“-2”（6个位0与4个位1）三种状况。

-   **有利于提取时钟**

    时钟恢复是依赖于“电平跳变沿“（后面会介绍），所以平衡”0“和”1“，可以简化了时钟恢复，降低了接收机成本。

-   **方便错误检查**

    8B/10B编码采用冗余方式，将8位的数据和一些特殊字符按照特定的规则编码成10位的数据，根据这些规则，能检测出传输过程中单个和多个比特误码。

有关8b/10b编码的编码算法可以进一步查阅相关文献。在SerDes中长常用编码方式除了8b/10b编码外，还有64b/66b编码等；

### 1.2 扰码

扰码是一种将数据重新排列或者进行编码以使其随机化的方法，但是必须能够解扰恢复。我们希望打乱长的连“0”和长的连“1”序列，将数据随机化。扰码产生是通过循环移位寄存器来实现的，而扰码生成多项式决定循环移位寄存器的结构。那么对信号加干扰有什么好处？

-   **解决EMI问题**

    当数据重复传输时，能量就会集中在某一频率上，产生EMI噪声；数据经过加扰后，能把集中的能量分散开，几乎变成白噪声。

-   **有利于提取时钟**

-   **同时又扩展了基带信号频谱，起到加密的效果**

总结到这里，看网友有提问，也提醒了我。问题是：在SerDes中先进行8b/10b编码之后再进行扰码，感觉两个的作用类似，只是扰码进一步增加“0”和“1”的随机性，还有没有其他更重要的作用？先加扰再编码有没有问题？初学者总有很多疑问。

### 1.3 串行器/解串器

串行器Serializer把并行信号转化为串行信号。Deserializer把串行信号转化为并行信号。一般地，并行信号为8 /10bit或者16/20bit宽度，串行信号为1bit宽度（也可以分阶段串行化，如8bit->4bit->2bit->equalizer->1bit以降低equalizer的工作频率）。

Serializer/Deserializer的实现采用双沿（DDR）的工作方式，利用面积换速度的策略，降低了电路中高频率电路的比例，从而降低了电路的噪声。

接收方向除了Deserializer之外，一般带有还有对齐功能逻辑（Aligner）。相对SerDes发送端，SerDes接收端起始工作的时刻是任意的，接收器正确接收的第一个 bit可能是发送数据的任意bit位置。因此需要对齐逻辑来判断从什么bit位置开始（哪里开始算是第一个数据），以组成正确的并行数据。对齐逻辑通过在串行数据流中搜索特征码字（Alignment Code）来决定串并转换的起始位置，这就需要用到了8b/10b的控制字符集，也就是我们常说的“K”字符，常见的什么K28.5、K28.0、K28.3、K28.4等，在JESD204B中他们又称/K/、/R/、/A/、/Q/字符。

### 1.4 前向反馈均衡（feed-forward equalization/equalizer）

SerDes信号从发送芯片到达接收芯片所经过的路径称为信道（channel），包括芯片封装、pcb走线、过孔、电缆、连接器等元件。从频域看，信道可以简化为一个低通滤波器（LPF）模型，如果SerDes的速率大于信道的截止频率，就会一定程度上损伤信号（高频被滤掉了，数字信号边沿会变得平滑）。均衡器的作用就是补偿信道对信号的损伤。

发送端的均衡器采用FFE（Feed forward equalizers）结构，从频域上看，FFE是一个高通滤波器（容易理解，信道损伤是一个低通滤波器，会抑制高频，那么均衡就补偿高频）。从时域上看，又叫加重器（emphasis）。加重分为去加重（de-emphasis）和预加重（pre-emphasis）。De-emphasis 降低差分信号的摆幅（swing）。Pre-emphasis增加差分信号的摆幅。由于目前芯片都需要追求降低功耗，所以大部分使用de-emphasis的方式，加重越强，信号的平均幅度会越小。

## 2. 接收端模块说明

### 2.2 接收均衡器

#### 2.2.1 线形均衡器(Linear Equalizer)

接收端均衡器的目标和发送均衡器是一致的。对于低速（<5Gbps）SerDes，通常采用连续时间域、线性均衡器实现如尖峰放大器（peaking amplifier）， 均衡器对高频分量的增益大于对低频分量的增益。

#### 2.2.2 裁决反馈均衡器（Decision Feedback Equalizer-DFE）

对于高速（>5Gbps）SerDes，由于信号的抖动（如ISI相关的确定性抖动）可能会超过或接近一个符号间隔（UI，Unit Interval），单单使用线性均衡器不再适用。线性均衡器对噪声和信号一起放大，并没有改善SNR或者说BER。对于高速SerDes，采用一种称作DFE的非线性均衡器。DFE通过跟踪过去多个UI的数据（history bits）来预测当前bit的采样门限，从而预测码间干扰。DFE只对信号放大，不对噪声放大，可以有效改善SNR。

不论是发送端还是接收端的均衡器，本质上都是高通滤波器，因为数字信号采样都希望边沿越抖越好，边沿变缓之后就会产生码间干扰。

如下图所示，系统传输一个“11011”的码流，如果没有均衡器，信号受到信道损伤，信号展宽，出现码间干扰，导致中间的“0”，无法被检测到。

DFE设计的关键是确定DFE系数，如果DFE的系数接近信道的脉冲相应，就可以到的比较理想的结果。但是信道是一个时变的媒介，比如温度电压工艺的慢变化等因素会改变信道的特性。因此DFE的系数需要自适应算法，自动扑获和跟随信道的变化。这也是实际应用中DFE功能开启的时候，做环境试验的结果很多时候会出问题，因为DFE在信道特性变化的时候，自适应的速度没有跟上。DFE系数自适应算法非常学术，每个厂商的算法都是保密的，不对外公布。

### 2.3 时钟数据恢复（CDR)

最开始接触SerDes的是，说“没有单独的时钟线，时钟嵌入在数据流中的”，脑海里第一反应就是在数据中插入一定规则的编码代表时钟的高电平或者低电平，回头一想，那时钟频率不就远低于数据的采集频率了吗？实际上，所谓的“时钟嵌在数据中”的意思，是时钟嵌在数据的跳变沿里。不难理解，极端情况下，假设一串数据流是"1"和"0"交替发送，那这不就是一个时钟了吗。

CDR（ Clock and Data Recovery）即时钟和数据的恢复，是SerDes设计中非常重要的环节。CDR常用的技术有基于数字锁相环（PLL）和基于相位插值器两种。当数据经过时，CDR就会捕获数据边沿跳变的频率，如果数据长时间没有跳变，CDR就无法得到精确的训练，CDR采样时刻就会漂移，可能采到比真实数据更多的“0”或“1”。这就是为什么我们在发送的时候采用8b/10b编码或扰码来避免重复出现“0”或“1”，原因之一也在此。所以CDR有一个指标叫做最长连“0”或连“1”长度容忍（Max Run Length或者Consecutive Identical Digits）能力。

恢复了时钟之后，需要恢复数据。第一步首先要将恢复出来的时钟与数据的边缘进行对齐，然后再将数据给读出来。在硬件原理上，使用PLL电路以及触发器即可。

以上是总结的有关SerDes的基础知识，希望能帮助像我一样的初学者建立初步的认识。SerDes每一个模式深入下去都有很多硬件、信号完整性（包括眼图的评价机制）的知识，想做到非常清晰的认识有一定的难度。目前，对于FPGA工程师来说，SerDes作为phy芯片或者集成在IP核中，先掌握应用，需要的时候再深入。



# 第四章：均衡技术
去加重  CTLE   FFE   DFE


# 第五章：时钟数据恢复技术综述

## 1. 技术概述与核心原理

**时钟数据恢复**（CDR，Clock and Data Recovery），又称时钟恢复或时钟再生，是一种从接收到的数据信号中提取出同步时钟信息，并以此时钟对数据进行重新采样和判定的关键技术。在高速通信系统中，发送端通常将数据和时钟信号编码在一起进行传输，以节省带宽。然而，信号在传输过程中会受到噪声、干扰和信道损耗的影响而发生畸变，导致接收端无法直接获得纯净的时钟。CDR技术的核心任务就是充当一位“修复师”，从失真的信号中分离并再生出原始的时钟和数据。

CDR的工作原理主要依赖于一个反馈控制系统，其核心通常是**锁相环**（PLL，Phase-Locked Loop）。一个典型的CDR架构包含以下三个关键模块：

- **鉴相器（PD，Phase Detector）**：比较输入数据信号的相位与本地时钟（由压控振荡器产生）的相位，输出一个代表两者相位差的误差信号。
- **环路滤波器（LF，Loop Filter）**：对鉴相器输出的误差信号进行低通滤波，平滑其中的高频噪声，产生一个稳定的控制电压。
- **压控振荡器（VCO，Voltage-Controlled Oscillator）**：根据环路滤波器输出的控制电压，动态调整其输出时钟的频率和相位，使其最终与输入数据流中的隐含时钟严格同步。

当环路锁定后，VCO输出的时钟便可作为最佳采样时钟，用于恢复原始数据。

## 2. 技术分类与演进

CDR技术可以根据其信号处理方式、电路架构和同步机制进行多维度的分类。

### 2.1 根据信号处理方式：A类与B类

这种分类方式揭示了CDR处理时钟信息的根本区别：

- **A类（开环）**：输出时钟直接受输入信号中检测到的时钟分量控制。其特点是电路相对简单，但当输入信号消失时，输出也随之消失。无源滤波方法属于此类。
- **B类（闭环）**：输出时钟通过锁相环间接地受输入信号的相位控制，与信号幅度无关。即使没有输入信号，压控振荡器仍会维持自由振荡输出。大多数基于锁相环的CDR都属于此类。

### 2.2 根据电路实现方式

1. **使用无源滤波器和限幅放大器**：典型的A类恢复方式，结构简单。
2. **使用窄频再生分频器**：特性介于A类和B类之间。
3. **使用同步振荡器**：特性更接近B类，利用振荡器的注入锁定特性。
4. **使用锁相环（PLL）**：最主流的B类实现方式，性能优越且可控性强。

### 2.3 根据同步机制：同步与异步架构

- **同步时钟恢复**：传统的锁相方式，旨在使本地时钟的相位精确同步于码元相位。缺点是锁定时间较慢。
- **异步时钟恢复**：基于插值原理和前馈控制的开环结构。它使用自由运行的本地时钟采样，然后通过数字信号处理（如定时误差估计和插值滤波器）计算出最佳采样时刻的值。这种方案捕获速度快，无需同步前导码，提高了信道效率。

### 2.4 模拟与数字CDR

根据实现域与工艺，可分为模拟与数字CDR。随着CMOS工艺的进步，**全数字CDR**（AD-CDR，All-Digital CDR）已成为主流。例如，近期有一款采用28nm CMOS工艺实现的数字CDR，其工作速率覆盖1.62至8.1 Gb/s，并能实现快速锁定和抗谐波锁定功能。数字架构具有面积小、功耗低、可移植性强、易于集成数字辅助功能等优点。

## 3. 关键性能指标

评估一个CDR性能的好坏，通常需要关注以下几个系统级的指标：

- **抖动产生**：指即使在没有输入抖动的情况下，CDR本身产生的时钟边沿的不确定性。它反映了CDR内部噪声的优劣。
- **抖动容限**：指CDR在保证不产生误码的前提下，能够容忍的输入信号上叠加的最大抖动。它衡量了CDR的抗干扰能力。
- **抖动传递**：描述了CDR输出时钟的抖动如何随输入信号的抖动变化，通常表现为一个低通滤波特性。锁相环的环路带宽决定了这个特性：带宽越宽，跟踪输入抖动越快，但抑制高频噪声能力越差。

## 4. 主要应用领域

CDR技术是现代电子系统不可或缺的一部分，其应用极为广泛：

- **高速串行总线**：在计算机内部，USB 3.0/3.1、PCI-Express（PCI-E）、SATA等接口都依赖CDR来从串行数据流中恢复时钟，实现高速可靠的数据传输。
- **光通信系统**：这是CDR的核心应用领域之一。从10Gbps到400Gbps乃至更高速率的光模块中，CDR是实现信号长距离、高质量传输的“隐形守护者”，有效克服色散和非线性效应带来的信号损伤。
- **存储系统**：在硬盘（HDD）的读写通道中，特别是采用部分响应最大似然（PRML）技术后，高性能的时钟恢复技术是实现高密度数据存储的关键。
- **测试与测量**：高速示波器、误码仪等测试设备内部也需要高精度的CDR电路，以对被测信号进行精确分析。

## 5. 前沿研究与未来趋势

面向未来的通信需求，CDR技术正朝着更高速率、更低功耗、更高集成度和更智能化的方向发展。

### 5.1 应对复杂信道环境的新算法

传统CDR在恶劣信道下可能失效。例如，在多孔径自由空间光通信中，强湍流会导致某些孔径信号深度衰落。针对此问题，研究者提出了基于**定时强度最大比合并**（TIMRC，Timing-Intensity Maximal Ratio Combining）的时钟恢复方案。实验证明，该算法能将时钟恢复精度提高0.78dB，收敛速度提升88.5%，并在大频偏下保持稳定，极大增强了系统的鲁棒性。

### 5.2 针对特定损伤的优化

在光纤通信中，色散会严重影响时钟恢复。近期研究提出了针对强度调制/直接检测系统的色散容忍时钟恢复算法，能有效应对非归零码和数字脉冲成型信号。

### 5.3 架构与电路的创新

无参考时钟CDR成为研究热点，以适应各种数据速率标准。新型数字CDR通过两阶段频率检测系统，实现了宽范围、无谐波锁定的快速频率捕获，适用于嵌入式DisplayPort等接口。下表总结了CDR技术的前沿研究方向与特点：

| 研究方向 | 关键技术/创新点 | 性能提升与应用场景 |
| :--- | :--- | :--- |
| **多通道与空间分集** | 定时强度最大比合并（TIMRC）算法 | 精度↑0.78dB，收敛速度↑88.5%，鲁棒性↑66%（多孔径FSO通信） |
| **数字域架构演进** | 全数字CDR，两阶段频率检测 | 宽速率范围（1.62-8.1Gb/s），快速锁定（<2μs），抗谐波锁定 |
| **算法与损伤补偿** | 色散容忍时钟恢复算法 | 应对色散影响，适用于IM/DD系统和PAM4调制 |
| **光域时钟恢复** | 光锁相环，无源光滤波（FBG） | 避免光电转换瓶颈，潜在的高速率处理能力 |
| **系统级与智能化** | 分布式同步算法，集成AI/ML | 提高多节点系统鲁棒性，智能优化CDR参数以适应复杂信道 |

## 6. 总结

展望未来，CDR技术不仅要应对单信道速率的持续攀升，还需在功耗和集成度上满足数据中心和移动设备的需求。同时，引入人工智能和机器学习技术来智能调节CDR参数，以最优状态适应动态变化的信道环境，也是一个充满潜力的发展方向。

# 第六章：DFX 
环回测试 等 


# 问题：SerDes中8b/10b编码与扰码的顺序与作用

8b/10b编码和扰码都是在比特层面进行操作，目的是改善信号质量。但它们就像“规矩的班长”和“随性的画家”，**解决问题的层次和手段截然不同**。

简单来说，**先8b/10b编码再加扰不仅没问题，反而是PCIe Gen1/2等经典高速总线的标准做法**。之所以这么设计，正是因为它们各司其职，且顺序至关重要。

---

## 🎯 核心区别：两种技术的核心差异

为了清晰地理解，我们可以通过一个表格来对比两者的核心差异：

| 维度 | 8b/10b 编码 | 扰码 (Scrambling) |
| :--- | :--- | :--- |
| **核心目的** | **保证DC平衡**，确保“0”和“1”数量基本相等，便于时钟恢复 | **能量扩散（频谱 shaping）**，将数据随机化，降低电磁干扰（EMI），避免频谱能量集中在特定频率 |
| **实现方式** | **通过查表**，将8位数据**严格映射**为特定的10位码字，这是一种**确定性的、一一对应**的规则 | **通过多项式（如LFSR）** 与数据进行异或运算，这是一种**概率性的**处理，使数据呈现伪随机特性 |
| **输出结果** | 输出码字中，“0”和“1”的个数差被严格限制（仅有0或±2），**最大游程长度不超过5位** | 理论上仍可能产生很长的连“0”或连“1”，只是**概率极低** |

### 形象类比
*   **8b/10b编码像一位严格的班长**：它通过既定的规则（映射表）强制每个10人小队里，男生和女生人数必须均衡，绝不允许出现全是男生或全是女生的极端情况。
*   **扰码像一位随性的画家**：它通过自己的算法（多项式）把原本有序的队伍（原始数据）打乱，让队伍的构成看起来更随机，从而避免在操场上形成过于规律的队形（频谱峰值），减少对周围的影响（EMI）。

---

## 🧩 为什么是这个顺序？先加扰再编码可以吗？

在你的SerDes设计中，先进行8b/10b编码，再进行扰码，这个顺序是经过深思熟虑的，原因如下：

1.  **扰码打不破8b/10b的“规矩”**：8b/10b编码器的输出，其“0”和“1”的个数差（即游程长度）已经被严格限定。在它**之后**进行的扰码，只是对这个已经“守规矩”的10位码字进行随机化处理。因为扰码的本质是异或运算，它可能会改变码字的具体内容，但**无法从根本上打破其内部“0”和“1”个数的均衡性**。也就是说，8b/10b辛苦建立的“纪律”（DC平衡）依然被保留。

2.  **若顺序颠倒，8b/10b会“抹去”扰码的成果**：如果反过来，**先对原始数据加扰，再进行8b/10b编码**。此时，8b/10b编码器看到的是一个已经被随机化的输入。为了完成自己的“职责”（保证DC平衡），它会**严格按照映射表**，将这个随机的8位输入转换成另一个特定的10位输出。这样一来，原本由扰码引入的随机性，就被8b/10b的确定性映射**完全覆盖和改变了**。输出信号中将不再保留加扰后的统计特性，扰码降低EMI的核心作用也就失效了。

---

## 📈 PCIe的演进就是最好的例证

*   **PCIe 1.0/2.0 (Gen1/Gen2)**：为了在2.5GT/s和5GT/s的速率下保证信号质量，采用了这种稳健的“**加扰 -> 8b/10b编码**”组合拳。
*   **PCIe 3.0 (Gen3)及以后**：当速率提升至8GT/s后，8b/10b编码20%的开销变得不可接受。因此，协议演进为 **128b/130b编码**。这种新编码只保留了极低的开销（约1.5%），但失去了内置的DC平衡能力。此时，**扰码的地位就从“辅助”变成了“核心”**，它必须独自承担起保证DC平衡和能量扩散的双重重任。

---

## 💡 总结

所以，你观察到的“先8b/10b编码再加扰”是完全正确的标准做法。它们不是功能重复，而是**在不同层次上解决了不同的问题**，并且这个顺序确保了两种机制能协同工作，而不是相互抵消。

# SerDes 系统中的“频域传输”：DMT 算法与 OFDM 的关联

当前的serdes系统，是否有将一段bit 转换到频域进行传输的。类似于NR的OFDM传输?
你的观察非常深刻。你所描述的“将 Bit 转换到频域传输”的技术，在高速有线传输领域确实存在，它被称为 **DMT (Discrete Multi-Tone，离散多信道调制)**。

它的原理几乎与 4G/5G 无线通信中的 **OFDM (正交频分复用)** 完全一致。

---

### 1. 为什么 SerDes 也要用“频域”传输？

传统的 SerDes（如跑 NRZ 或 PAM4 编码的）属于**单载波时域传输**。但随着速率跨入 **224Gbps** 甚至更高，链路面临两个致命问题：

* **高频衰减极度严重：** 铜缆或 PCB 在 50GHz 以上的损耗可能超过 **40-50dB**，像一堵墙一样挡住了信号。
* **频谱凹陷 (Notch)：** 阻抗不连续（如过孔或连接器）会产生特定的“频率死角”，单载波信号如果撞上这个死角，整个信道能量就会大幅坍塌。

**DMT 算法的解法（类 OFDM）：**
它通过 **IFFT（快速傅里叶逆变换）** 把高速比特流分散到成百上千个小的**子载波（Sub-carriers）**上。
* **避重就轻：** 算法可以探测哪些频率点损耗小，就在那里多放数据（采用高阶 QAM 调制）；哪些频率点损耗大或有死角，就少放甚至不放数据。这被称为 **Bit Loading（比特加载）**。



---

### 2. 既然这么好，为什么主流 SerDes 还是用时域（PAM4）？

虽然 DMT/OFDM 在理论上频谱效率极高，但在有线 SerDes（特别是集成在芯片内部的 IP，如 HiLink）中，它面临巨大的工程挑战：

1.  **峰均比 (PAPR) 太高：** 频域叠加后的波形会有很高的瞬时尖峰。为了不让信号失真，驱动电路必须在线性区工作，这会导致 **Tx 端的功耗激增**，对于散热压力巨大的数据中心 ASIC 芯片是致命的。
2.  **延迟 (Latency) 太大：** OFDM 需要做 FFT/IFFT 转换，还要加循环前缀（CP），这会带来几十到上百纳秒的延迟。对于 CPU 到内存、或超算集群内部的极速交换，这个延迟太长了。
3.  **DSP 复杂度：** 在 224G 这种速率下做高精度的浮点 FFT 运算，芯片面积和动态功耗远高于时域的 FFE/DFE 均衡器。

---

### 3. 当前的实际应用现状

虽然在**传统的板级 SerDes** 中，时域（PAM4 + DSP）仍是霸主，但在以下领域，你说的“频域传输”已经是主流或强力竞争者：

* **相干光通信 (Coherent Optical)：** 由于传输距离极长（上千公里）且色散严重，频域均衡和类似 OFDM 的调制是标准配置。
* **超高速电缆 (DAC)：** 针对超过 200Gbps 的铜缆传输，行业正在激烈讨论是否引入 DMT 来替代 PAM4，以榨取铜缆最后的物理带宽。
* **DSL/窄带通信：** 比如早期的 ADSL/VDSL 宽带，本质上就是一种运行在较低频率、完全基于 DMT 算法的“慢速 SerDes”。

---

### 4. 总结对比：时域 vs 频域

| 特性 | 时域 SerDes (PAM4) | 频域 SerDes (DMT/OFDM) |
| :--- | :--- | :--- |
| **类似技术** | 类似于单载波码分 | **类似于 5G NR / Wi-Fi** |
| **优势** | 延迟低、电路简单、功耗相对可控 | **抗频率选择性衰减强、频谱利用率极高** |
| **劣势** | 遇到频谱死角（Notch）时表现差 | **PAPR 高、计算量巨大、延迟高** |
| **当前状态** | **112G/224G 绝对主流** | 224G+ 实验阶段 / 相干光通信常用 |



---

**一句话总结：**
你提到的“频域传输”在 SerDes 领域主要体现为 **DMT**。它虽然在抗损耗上是“降维打击”，但因为**太费电**和**太慢（高延迟）**，目前还没能全面取代像 **HiLink** 这样基于时域均衡的

# 揭秘 "FIRWare"：SerDes 中的 FIR 算法与固件控制

在 SerDes（如华为 HiLink、TSMC IP）的讨论中，你听到的类似“FIRWare”的术语，核心指向的是 **Tx 端（发送端）的数字预加重算法及其自动配置机制**。

---

### 1. 核心组成：FIR + Firmware

* **FIR (Finite Impulse Response)：** 这是硬件层面的滤波器（由多个 **Taps 抽头**组成）。它的物理作用是通过调整信号波形的“峰值”（Pre-shoot/Over-shoot）来对抗传输线的损耗。
* **Firmware (固件)：** 这是运行在 SerDes 控制微核（如小 RISC-V 或状态机）上的软件代码。它的作用是计算出“当前环境下，FIR 应该设为多少”。

**所以，“FIRWare”在工程语境下是指：运行在芯片内部，用来自动寻找并写入最佳 FIR 抽头系数（C-1, C0, C+1）的那套算法程序。**



---

### 2. FIR 算法在其中干了什么？（以 3-Tap 为例）

FIR 算法通过操纵电压，人为制造“信号峰值”：

1.  **Main Tap (C0)：** 决定稳态电平的基本强度。
2.  **Pre-cursor (C-1)：** 在信号跳变**之前**制造一个尖峰。它能抵消链路的高频相位失真。
3.  **Post-cursor (C+1)：** 在信号跳变**之后**制造一个尖峰。它能快速“拉回”拖尾信号，消除码间干扰（ISI）。



---

### 3. 为什么需要“固件”参与？（自动化的代价）

如果你手动调节 FIR 参数，针对不同的光纤长度、温度、工艺偏快或偏慢（PVT），你可能需要尝试上万种组合。

**“FIR 算法固件”的功能：**
* **Auto-Tuning (自动调优)：** 固件会根据接收端反馈回来的眼图质量数据，实时调整 Tx 端的 FIR 系数。
* **Look-up Table (查找表)：** 固件内预存了不同链路（如 100米光纤 vs 2米铜缆）的经验参数，实现“一键适配”。
* **协议协同：** 在 PCIe 5.0/6.0 的 **Link Training** 阶段，固件负责在几十毫秒内完成与对端芯片的 FIR 系数握手。

---

### 4. 总结对比

| 概念 | 侧重点 | 形象比喻 |
| :--- | :--- | :--- |
| **FIR (硬件)** | 物理执行机构 | 汽车的油门和刹车踏板 |
| **Firmware (软件)** | 控制策略和算法 | 汽车的自动巡航系统（自动踩油门） |
| **"FIRWare" (综合体)** | **闭环自动补偿能力** | **整套自动驾驶算法** |

---

**一句话总结：**
当你听到“FIRWare”时，你应该理解为 **“发送端那个能根据链路损耗自动调节信号峰值的智能算法”**。

# 参考文章：
https://zhuanlan.zhihu.com/p/671755097?share_code=4a5vE02qtLrF&utm_psn=2013356126469637159